<a href="https://colab.research.google.com/github/AlperYildirim1/Language-as-Waves/blob/main/FNet_Train_Last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torchmetrics sacrebleu x-transformers

## CONFIG

In [ ]:
!pip install -q torchmetrics sacrebleu x-transformers

## CONFIG

# --- Data & Task Size ---
MAX_LENGTH = 128

MODEL_CHOICE = "FFNet-A100-80GB-v4-high-precision-fixed" # Renamed for clarity

# --- Model Architecture Config ---
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048
DROPOUT = 0.1

# --- Layer counts ---
NUM_ENCODER_LAYERS = 7
NUM_DECODER_LAYERS = 6

# --- Training Config (ADJUSTED FOR FAIR COMPARISON) ---

TARGET_TRAINING_STEPS = 100000
GRAD_ACCUMULATION_STEPS = 2


VALIDATION_SCHEDULE = [
    2000, 4000, 5000, 7500, 10000, 15000, 20000,
    25000, 30000, 35000, 42500, 50000, 57500, 65000, 72500, 90000, 100000
]
PEAK_LEARNING_RATE = 6e-4
WARMUP_STEPS = 600 # Warmup can stay similar or scale slightly, 600 is fine
WEIGHT_DECAY = 0.01

# --- Regularization Config ---
LABEL_SMOOTHING_EPSILON = 0.1

# --- Other Constants ---
DRIVE_BASE_PATH = "/content/drive/MyDrive/AIAYN"
ORIGINAL_BUCKETED_REPO_ID = "Yujivus/wmt14-de-en-bucketed-w4" # Use the bucketed one (we will ignore buckets)
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-de-en"

## DATALOADERS

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset
import math
import os
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter
import random
import numpy as np
import torch
from transformers import get_cosine_schedule_with_warmup
from typing import List
from transformers import AutoModel
from transformers import DataCollatorForSeq2Seq


def set_seed(seed_value=5):
    """Sets the seed for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 117
set_seed(SEED)
print(f"Reproducibility seed set to {SEED}")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

#torch.use_deterministic_algorithms(True)

print("--- Loading Modernized Configuration ---")
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

torch.set_float32_matmul_precision('high')
print("✅ PyTorch matmul precision set to 'high'")

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

VOCAB_SIZE = len(tokenizer)
print(f"Vocab size: {VOCAB_SIZE}")


# DATA LOADING & PREPARATION

# --- 1. DEFINE THE FNET COLLATOR (FORCE FIXED LENGTH) ---
# This is crucial. It forces every sentence to be exactly 128 tokens.
fnet_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding="max_length",    # <--- FORCE PADDING
    max_length=MAX_LENGTH,   # <--- 128 (defined in your config)
    pad_to_multiple_of=None
)

# --- 2. LOAD DATASET ---
print(f"Loading original bucketed samples from: {ORIGINAL_BUCKETED_REPO_ID}")
original_datasets = load_dataset(ORIGINAL_BUCKETED_REPO_ID)

# --- 3. CREATE DATALOADERS (STANDARD FIXED SIZE) ---
FNET_PHYSICAL_BATCH_SIZE = 320

g = torch.Generator()
g.manual_seed(SEED)

train_dataloader = DataLoader(
    original_datasets["train"],
    batch_size=FNET_PHYSICAL_BATCH_SIZE,  # <--- FIXED BATCH SIZE (Safe from OOM)
    shuffle=True,                # <--- GLOBAL SHUFFLE
    num_workers=8,
    collate_fn=fnet_collator,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
)

val_dataloader = DataLoader(
    original_datasets["validation"],
    batch_size=FNET_PHYSICAL_BATCH_SIZE,
    collate_fn=fnet_collator,
    num_workers=8,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
)

print(f"Train Dataloader is now a STANDARD iterator.")
print(f"Physical Batch Size: {FNET_PHYSICAL_BATCH_SIZE}")
print(f"Gradient Accumulation: {GRAD_ACCUMULATION_STEPS}")
print(f"Effective Batch Size: {FNET_PHYSICAL_BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")

# --- SANITY CHECK ---
print("\n--- Running Sanity Check on new FNet DataLoader ---")
train_dataloader.generator.manual_seed(SEED)
temp_iterator = iter(train_dataloader)
print("Shapes of first 3 batches (Should all be [64, 128]):")
for i in range(3):
    batch = next(temp_iterator)
    print(f"  Batch {i+1}: input_ids shape = {batch['input_ids'].shape}")
print("--- Sanity Check Complete ---\n")
# --- VERIFY SHUFFLE IS WORKING ---
print("🕵️ INSPECTING ONE BATCH 🕵️")

# Get one batch from your active train_dataloader
batch = next(iter(train_dataloader))
input_ids = batch['input_ids']

# Calculate real lengths (ignoring padding)
# We count how many tokens are NOT the pad token (usually 0 or 58100)
real_lengths = (input_ids != tokenizer.pad_token_id).sum(dim=1)

print(f"Batch Shape: {input_ids.shape}")
print("Random Sample of 20 lengths in this batch:")
print(real_lengths[:20].tolist())

# Check diversity
if real_lengths.float().std() < 5:
    print("\n⚠️ WARNING: LENGTHS LOOK CLUSTERED! (Bad shuffling)")
else:
    print(f"\n✅ PASSED: Lengths are highly variable (Std Dev: {real_lengths.float().std():.2f}). Shuffling is working.")

##  Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from x_transformers import Encoder, Decoder

class RoPETransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, num_heads, d_model, dff, vocab_size, max_length, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

        # We REMOVE self.pos_encoder (RoPE handles position internally)
        self.dropout_layer = nn.Dropout(dropout)

        # --- x-transformers Encoder ---
        self.encoder = Encoder(
            dim = d_model,
            depth = num_encoder_layers,
            heads = num_heads,
            attn_dim_head = d_model // num_heads,
            ff_mult = dff / d_model,
            rotary_pos_emb = True,
            attn_flash = True,
            attn_dropout = dropout,
            ff_dropout = dropout,
            use_rmsnorm = True
        )

        # --- x-transformers Decoder ---
        self.decoder = Decoder(
            dim = d_model,
            depth = num_decoder_layers,
            heads = num_heads,
            attn_dim_head = d_model // num_heads,
            ff_mult = dff / d_model,
            rotary_pos_emb = True,
            cross_attend = True,
            attn_flash = True,
            attn_dropout = dropout,
            ff_dropout = dropout,
            use_rmsnorm = True
        )

        self.final_linear = nn.Linear(d_model, vocab_size)
        self.final_linear.weight = self.embedding.weight

    def forward(self, src, tgt, src_padding_mask, tgt_padding_mask, memory_key_padding_mask, tgt_mask):
        # 1. Embeddings (No Absolute Positional Encoding added!)
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        src_emb = self.dropout_layer(src_emb)

        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.dropout_layer(tgt_emb)

        # 2. Mask Conversion
        # User provides True=PAD. x-transformers wants True=KEEP.
        # We invert the boolean mask using ~
        enc_mask = ~src_padding_mask if src_padding_mask is not None else None
        dec_mask = ~tgt_padding_mask if tgt_padding_mask is not None else None

        # Note: 'tgt_mask' (causal mask) is handled automatically by x-transformers Decoder!
        # We do NOT pass the square causal mask manually.

        # 3. Encoder
        # x-transformers takes embeddings directly
        memory = self.encoder(src_emb, mask=enc_mask)

        # 4. Decoder
        # context = memory (from encoder)
        # context_mask = mask for memory (encoder mask)
        decoder_output = self.decoder(
            tgt_emb,
            context=memory,
            mask=dec_mask,
            context_mask=enc_mask
        )

        return self.final_linear(decoder_output)

    # Keep your existing create_masks (used for Data Processing mostly)
    def create_masks(self, src, tgt):
        src_padding_mask = (src == tokenizer.pad_token_id)
        tgt_padding_mask = (tgt == tokenizer.pad_token_id)
        # We still generate this for compatibility, though x-transformers handles causality internally
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            sz=tgt.size(1), device=src.device, dtype=torch.bool
        )
        return src_padding_mask, tgt_padding_mask, src_padding_mask, tgt_mask

    @torch.no_grad()
    def generate(self, src: torch.Tensor, max_length: int, num_beams: int = 5) -> torch.Tensor:
        self.eval()
        # Create Mask (True=PAD)
        src_padding_mask = (src == tokenizer.pad_token_id)
        # Invert for x-transformers (True=KEEP)
        enc_mask = ~src_padding_mask

        # Encode
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        # No Pos Encoder
        memory = self.encoder(self.dropout_layer(src_emb), mask=enc_mask)

        batch_size = src.shape[0]
        # Expand for beams
        memory = memory.repeat_interleave(num_beams, dim=0)
        enc_mask = enc_mask.repeat_interleave(num_beams, dim=0)

        initial_token = tokenizer.pad_token_id
        beams = torch.full((batch_size * num_beams, 1), initial_token, dtype=torch.long, device=src.device)
        beam_scores = torch.zeros(batch_size * num_beams, device=src.device)
        finished_beams = torch.zeros(batch_size * num_beams, dtype=torch.bool, device=src.device)

        for _ in range(max_length - 1):
            if finished_beams.all(): break

            # Embed beams
            tgt_emb = self.embedding(beams) * math.sqrt(self.d_model)
            # No Pos Encoder

            # Decode
            # x-transformers automatically handles the causal masking for the sequence length of tgt_emb
            decoder_output = self.decoder(
                self.dropout_layer(tgt_emb),
                context=memory,
                context_mask=enc_mask
            )

            logits = self.final_linear(decoder_output[:, -1, :])
            log_probs = F.log_softmax(logits, dim=-1)

            # ... (Rest of your Beam Search Logic remains identical) ...
            log_probs[:, tokenizer.pad_token_id] = -torch.inf
            if finished_beams.any(): log_probs[finished_beams, tokenizer.eos_token_id] = 0

            total_scores = beam_scores.unsqueeze(1) + log_probs
            if _ == 0:
                total_scores = total_scores.view(batch_size, num_beams, -1)
                total_scores[:, 1:, :] = -torch.inf
                total_scores = total_scores.view(batch_size * num_beams, -1)
            else:
                total_scores = beam_scores.unsqueeze(1) + log_probs

            total_scores = total_scores.view(batch_size, -1)
            top_scores, top_indices = torch.topk(total_scores, k=num_beams, dim=1)

            beam_indices = top_indices // log_probs.shape[-1]
            token_indices = top_indices % log_probs.shape[-1]

            batch_indices = torch.arange(batch_size, device=src.device).unsqueeze(1)
            effective_indices = (batch_indices * num_beams + beam_indices).view(-1)

            beams = beams[effective_indices]
            beams = torch.cat([beams, token_indices.view(-1, 1)], dim=1)
            beam_scores = top_scores.view(-1)
            finished_beams = finished_beams | (beams[:, -1] == tokenizer.eos_token_id)

        final_beams = beams.view(batch_size, num_beams, -1)
        final_scores = beam_scores.view(batch_size, num_beams)
        normalized_scores = final_scores / (final_beams != tokenizer.pad_token_id).sum(-1).float().clamp(min=1)
        best_beams = final_beams[torch.arange(batch_size), normalized_scores.argmax(1), :]
        self.train()
        return best_beams

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # 1. Calculate the mean of the squares
        mean_square = x.pow(2).mean(dim=-1, keepdim=True)

        # 2. Calculate the inverse square root (1 / RMS)
        #    We add eps before the sqrt for stability
        inv_rms = torch.rsqrt(mean_square + self.eps)

        # 3. Normalize and scale
        return x * inv_rms * self.gamma


class FNetBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm_mix = nn.LayerNorm(d_model) # LayerNorm is safer for FNet than RMSNorm
        self.norm_ff = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # 1. Fourier Mixing Branch
        residual = x
        x = self.norm_mix(x)

        # --- THE FIX ---
        with torch.cuda.amp.autocast(enabled=False):
            x = x.float()
            # norm='ortho' makes the FFT energy-preserving.
            # Output magnitude will match input magnitude (~1).
            x = torch.fft.fftn(x, dim=(-2, -1), norm='ortho').real
            x = x.to(dtype=residual.dtype)
        # ---------------

        # Now 'x' and 'residual' have roughly same magnitude.
        # The skip connection works again.
        x = x + residual

        # 2. Feed Forward Branch
        residual = x
        x = self.norm_ff(x)
        x = self.ff(x)
        return x + residual


class FNetEncoder(nn.Module):
    def __init__(self, depth, d_model, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            FNetBlock(d_model, d_ff, dropout) for _ in range(depth)
        ])
        # [FIX] Use LayerNorm here to match the blocks
        self.norm_out = nn.LayerNorm(d_model)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm_out(x)

# --- Main Hybrid Model ---

class FNetHybridTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, num_heads, d_model, dff, vocab_size, max_length, dropout):
        super().__init__()
        self.d_model = d_model

        # Shared Embeddings
        # padding_idx=tokenizer.pad_token_id forces the vector at this index to be strict ZEROS.
        # It does not have gradients, it stays zero forever.
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=tokenizer.pad_token_id)

        # FNet REQUIRES Absolute Positional Embeddings because FFT mixes information
        # but doesn't inherently understand sequence order like RoPE/RNNs do initially.
        self.pos_embedding = nn.Embedding(max_length, d_model)

        self.dropout_layer = nn.Dropout(dropout)

        # --- Custom FNet Encoder ---
        self.encoder = FNetEncoder(
            depth=num_encoder_layers,
            d_model=d_model,
            d_ff=dff,
            dropout=dropout
        )

        # --- x-transformers Decoder (Retains RoPE) ---
        self.decoder = Decoder(
            dim=d_model,
            depth=num_decoder_layers,
            heads=num_heads,
            attn_dim_head=d_model // num_heads,
            ff_mult=dff / d_model,
            rotary_pos_emb=True,     # Decoder still uses RoPE
            cross_attend=True,
            attn_flash=True,
            attn_dropout=dropout,
            ff_dropout=dropout,
            use_rmsnorm=True
        )

        self.final_linear = nn.Linear(d_model, vocab_size)
        self.final_linear.weight = self.embedding.weight

    def forward(self, src, tgt, src_padding_mask, tgt_padding_mask, memory_key_padding_mask, tgt_mask):
        # 1. Embeddings
        # Source (Encoder) gets Absolute Positional Embeddings
        B, L_src = src.shape
        pos_ids = torch.arange(L_src, device=src.device).unsqueeze(0)
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        src_emb = src_emb + self.pos_embedding(pos_ids)
        src_emb = self.dropout_layer(src_emb)

        # Target (Decoder) gets NO Positional Embeddings here (RoPE handles it inside Decoder)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.dropout_layer(tgt_emb)

        # 2. Prepare Masks
        # x-transformers requires True = Keep, False = Mask
        # Your dataloader provides True = Pad
        enc_mask = ~src_padding_mask if src_padding_mask is not None else None
        dec_mask = ~tgt_padding_mask if tgt_padding_mask is not None else None

        # 3. FNet Encoder
        # Note: FNet mixes ALL tokens (including padding).
        memory = self.encoder(src_emb)

        # CRITICAL: Zero out padding positions in encoder output so Decoder doesn't attend to them.
        if src_padding_mask is not None:
            memory = memory.masked_fill(src_padding_mask.unsqueeze(-1), 0.0)

        # 4. RoPE Decoder
        # The decoder uses RoPE for self-attention on 'tgt',
        # and standard cross-attention to 'memory' (FNet output).
        decoder_output = self.decoder(
            tgt_emb,
            context=memory,
            mask=dec_mask,
            context_mask=enc_mask
        )

        return self.final_linear(decoder_output)

    def create_masks(self, src, tgt):
        # Standard mask creation (Same as your original)
        src_padding_mask = (src == tokenizer.pad_token_id)
        tgt_padding_mask = (tgt == tokenizer.pad_token_id)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            sz=tgt.size(1), device=src.device, dtype=torch.bool
        )
        return src_padding_mask, tgt_padding_mask, src_padding_mask, tgt_mask

    @torch.no_grad()
    def generate(self, src: torch.Tensor, max_length: int, num_beams: int = 5) -> torch.Tensor:
        self.eval()
        B, L_src = src.shape

        # 1. Encode with FNet
        pos_ids = torch.arange(L_src, device=src.device).unsqueeze(0)
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        src_emb = src_emb + self.pos_embedding(pos_ids)

        memory = self.encoder(self.dropout_layer(src_emb))

        # Masking padding in memory
        src_padding_mask = (src == tokenizer.pad_token_id)
        memory = memory.masked_fill(src_padding_mask.unsqueeze(-1), 0.0)

        # Prepare for Decoder (x-transformers style mask: True=Keep)
        enc_mask = ~src_padding_mask

        # --- BEAM SEARCH SETUP ---
        # Expand memory for beams
        memory = memory.repeat_interleave(num_beams, dim=0)
        enc_mask = enc_mask.repeat_interleave(num_beams, dim=0)

        initial_token = tokenizer.pad_token_id
        beams = torch.full((B * num_beams, 1), initial_token, dtype=torch.long, device=src.device)
        beam_scores = torch.zeros(B * num_beams, device=src.device)
        finished_beams = torch.zeros(B * num_beams, dtype=torch.bool, device=src.device)

        for _ in range(max_length - 1):
            if finished_beams.all(): break

            # Decoder Step (RoPE handled internally)
            tgt_emb = self.embedding(beams) * math.sqrt(self.d_model)

            decoder_output = self.decoder(
                self.dropout_layer(tgt_emb),
                context=memory,
                context_mask=enc_mask
            )

            logits = self.final_linear(decoder_output[:, -1, :])
            log_probs = F.log_softmax(logits, dim=-1)

            # --- STANDARD BEAM LOGIC (No changes needed here) ---
            log_probs[:, tokenizer.pad_token_id] = -torch.inf
            if finished_beams.any(): log_probs[finished_beams, tokenizer.eos_token_id] = 0

            total_scores = beam_scores.unsqueeze(1) + log_probs
            if _ == 0:
                total_scores = total_scores.view(B, num_beams, -1)
                total_scores[:, 1:, :] = -torch.inf
                total_scores = total_scores.view(B * num_beams, -1)
            else:
                total_scores = beam_scores.unsqueeze(1) + log_probs

            total_scores = total_scores.view(B, -1)
            top_scores, top_indices = torch.topk(total_scores, k=num_beams, dim=1)

            beam_indices = top_indices // log_probs.shape[-1]
            token_indices = top_indices % log_probs.shape[-1]

            batch_indices = torch.arange(B, device=src.device).unsqueeze(1)
            effective_indices = (batch_indices * num_beams + beam_indices).view(-1)

            beams = beams[effective_indices]
            beams = torch.cat([beams, token_indices.view(-1, 1)], dim=1)
            beam_scores = top_scores.view(-1)
            finished_beams = finished_beams | (beams[:, -1] == tokenizer.eos_token_id)

        final_beams = beams.view(B, num_beams, -1)
        final_scores = beam_scores.view(B, num_beams)
        normalized_scores = final_scores / (final_beams != tokenizer.pad_token_id).sum(-1).float().clamp(min=1)
        best_beams = final_beams[torch.arange(B), normalized_scores.argmax(1), :]
        self.train()
        return best_beams

In [ ]:
def count_parameters(model):
    table_data = []
    total_params = 0
    trainable_params = 0

    # 1. Global Counts
    for p in model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()

    print("="*40)
    print(f"📊 MODEL STATISTICS")
    print("="*40)
    print(f"Total Parameters:     {total_params:,}  ({total_params/1e6:.2f}M)")
    print(f"Trainable Parameters: {trainable_params:,}  ({trainable_params/1e6:.2f}M)")
    print("-" * 40)

    # 2. Section Breakdown
    def get_params(module):
        return sum(p.numel() for p in module.parameters())

    if hasattr(model, 'encoder'):
        enc_p = get_params(model.encoder)
        print(f"  • Encoder (FNet):   {enc_p:,}  ({enc_p/1e6:.2f}M)")

    if hasattr(model, 'decoder'):
        dec_p = get_params(model.decoder)
        print(f"  • Decoder (RoPE):   {dec_p:,}  ({dec_p/1e6:.2f}M)")

    if hasattr(model, 'embedding'):
        emb_p = get_params(model.embedding)
        print(f"  • Embeddings:       {emb_p:,}  ({emb_p/1e6:.2f}M)")

    print("="*40)



## Functions (Loss, Eval etc)

In [ ]:

translation_loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100,  # We don't calculate loss for pad tokens. Pad tokens are replaced with -100 by DataCollatorForSeq2Seq.
    label_smoothing=LABEL_SMOOTHING_EPSILON
)
def calculate_combined_loss(model_outputs, target_labels):
    """Calculates the loss based on the model's output structure."""
    logits = model_outputs
    translation_loss = translation_loss_fn(logits.reshape(-1, logits.shape[-1]), target_labels.reshape(-1))
    loss_dict = {'total': translation_loss.item()}
    return translation_loss, loss_dict

from torchmetrics.text import SacreBLEUScore

def evaluate(model, dataloader, device):
    # Use SacreBLEUScore (defaults to '13a' tokenizer, the WMT standard)
    metric = SacreBLEUScore().to(device)

    model.eval()

    # Use no_grad to save memory and speed up validation
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels']

            # Generate predictions
            generated_ids = model.generate(input_ids, max_length=MAX_LENGTH, num_beams=5)

            # Decode predictions
            pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            # Decode labels (Fixing -100 padding)
            labels[labels == -100] = tokenizer.pad_token_id
            ref_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

            # Update Metric
            # SacreBLEU expects references as a list of lists: [[ref1], [ref2], ...]
            formatted_refs = [[ref] for ref in ref_texts]
            metric.update(pred_texts, formatted_refs)

    model.train()

    # Compute returns a tensor, .item() converts it to a standard python float
    return metric.compute().item()



## WARNING! THIS CAN'T BE USED FOR FNET
def generate_sample_translations(model, device, sentences_de):
    """Generates and prints sample translations using beam search."""
    print("\n--- Generating Sample Translations (with Beam Search) ---")
    orig_model = getattr(model, '_orig_mod', model)
    orig_model.eval()

    inputs = tokenizer(sentences_de, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
    input_ids = inputs.input_ids.to(device)
    generated_ids = orig_model.generate(input_ids, max_length=MAX_LENGTH, num_beams=5)

    translations = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    for src, out in zip(sentences_de, translations):
        print(f"  DE Source: {src}")
        print(f"  EN Output: {out}")
        print("-" * 20)
    orig_model.train()

sample_sentences_de_for_tracking = [
    "Eine Katze sitzt auf der Matte.",
    "Ein Mann in einem roten Hemd liest ein Buch.",
    "Was ist die Hauptstadt von Deutschland?",
    "Ich gehe ins Kino, weil der Film sehr gut ist.",
]

def init_other_linear_weights(m):
    if isinstance(m, nn.Linear):
        # The 'is not' check correctly skips the final_linear layer,
        # leaving its weights tied to the correctly initialized embeddings.
        if m is not getattr(model, '_orig_mod', model).final_linear:
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)




In [ ]:
import json
import os
import subprocess
import torch
import hashlib
import sys
import shutil

# This logger will be configured and used in the main training script
import logging
logger = logging.getLogger(__name__)


def log_to_run_specific_file(run_dir):
    run_log_path = os.path.join(run_dir, "run_log.txt")
    file_handler = logging.FileHandler(run_log_path)
    file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
    logger.addHandler(file_handler)
    return file_handler

def log_configurations(log_dir, config_vars):
    # (Same as your provided function)
    config_path = os.path.join(log_dir, "config.json")
    try:
        with open(config_path, 'w') as f:
            serializable_configs = {k: v for k, v in config_vars.items() if isinstance(v, (int, float, str, bool, list, dict, type(None)))}
            json.dump(serializable_configs, f, indent=4)
        logger.info(f"Configurations saved to {config_path}")
    except Exception as e:
        logger.error(f"Could not save configurations: {e}")

def log_environment(log_dir):
    # (Same as your provided function)
    env_path = os.path.join(log_dir, "environment.txt")
    try:
        with open(env_path, 'w') as f:
            f.write(f"--- Timestamp (UTC): {datetime.datetime.utcnow().isoformat()} ---\n")
            f.write(f"Python Version: {sys.version}\n")
            f.write(f"PyTorch Version: {torch.__version__}\n")
            f.write(f"CUDA Available: {torch.cuda.is_available()}\n")
            if torch.cuda.is_available():
                f.write(f"CUDA Version: {torch.version.cuda}\n")
                f.write(f"CuDNN Version: {torch.backends.cudnn.version()}\n")
                f.write(f"Number of GPUs: {torch.cuda.device_count()}\n")
                f.write(f"GPU Name: {torch.cuda.get_device_name(0)}\n")
            f.write("\n--- Full pip freeze ---\n")
            result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=subprocess.PIPE, text=True, check=True)
            f.write(result.stdout)
        logger.info(f"Environment info saved to {env_path}")
    except Exception as e:
        logger.error(f"Could not save environment info: {e}")

def log_code_snapshot(log_dir, script_path):
    # NOTE: In Colab, you must save your notebook as a .py file for this to work.
    # For example, file -> "Save a copy as .py"
    code_dir = os.path.join(log_dir, "code_snapshot")
    os.makedirs(code_dir, exist_ok=True)
    if script_path and os.path.exists(script_path):
        try:
            shutil.copy(script_path, os.path.join(code_dir, os.path.basename(script_path)))
            logger.info(f"Copied script '{script_path}' to snapshot directory for verification.")
        except Exception as e:
            logger.error(f"Could not copy script for snapshot: {e}")
    else:
        logger.warning(f"Code Snapshot: Script path '{script_path}' not found. SKIPPING.")

def get_file_hash(filepath):
    # (Same as your provided function)
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception as e:
        logger.error(f"Could not generate hash for {filepath}: {e}")
        return None

def create_checksum_file(run_dir, artifacts_dict):
    checksum_file_path = os.path.join(run_dir, "checksums.sha256")
    logger.info(f"--- Creating digital fingerprints for key artifacts ---")
    with open(checksum_file_path, "w") as f:
        f.write(f"SHA256 Checksums for run: {os.path.basename(run_dir)}\n")
        for name, path in artifacts_dict.items():
            if path and os.path.exists(path):
                file_hash = get_file_hash(path)
                if file_hash:
                    log_message = f"  - {name} ({os.path.basename(path)}): {file_hash}"
                    logger.info(log_message)
                    f.write(f"{file_hash}  {os.path.basename(path)}\n")
            else:
                logger.warning(f"  - Skipped hashing '{name}', file not found: {path}")
    logger.info(f"Checksums saved to {checksum_file_path}")

def init_weights_kaiming(m):
    """
    Applies Kaiming He initialization to Linear layers.
    This is the standard, superior way to initialize deep Transformers.
    NOTE: We will handle the Embedding layer separately.
    """

    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5)) # a=sqrt(5) mimics default PyTorch for LeakyReLU
        if m.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(m.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(m.bias, -bound, bound)


def init_weights_fnet(m):
    """
    Specific initialization for FNet Hybrid.
    FNet is essentially a BERT-like encoder, so we use BERT-style initialization
    (Truncated Normal or Xavier) rather than Kaiming.
    """
    if isinstance(m, nn.Linear):
        # Xavier (Glorot) Uniform is the standard for Transformer/FNet attention/FFN layers
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

    elif isinstance(m, nn.Embedding):
        # Critical: Keep embedding variance low (0.02)
        nn.init.normal_(m.weight, mean=0.0, std=0.02)

    # Handle the RMSNorms if they have learnable parameters
    elif isinstance(m, (nn.LayerNorm, RMSNorm)):
        if hasattr(m, 'weight') and m.weight is not None:
            nn.init.ones_(m.weight)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.zeros_(m.bias)



## Training Loop

In [ ]:
if __name__ == '__main__':

    experiment_name = f"{MODEL_CHOICE}"
    CURRENT_RUN_DIR = os.path.join(DRIVE_BASE_PATH, experiment_name)
    SAVE_DIR = os.path.join(CURRENT_RUN_DIR, "models")
    LOG_DIR_TENSORBOARD = os.path.join(CURRENT_RUN_DIR, "tensorboard_logs")
    LOG_FILE_TXT = os.path.join(CURRENT_RUN_DIR, "run_log.txt")

    os.makedirs(SAVE_DIR, exist_ok=True)
    os.makedirs(LOG_DIR_TENSORBOARD, exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(message)s',
        handlers=[logging.FileHandler(LOG_FILE_TXT), logging.StreamHandler(sys.stdout)],
        force=True
    )
    logger = logging.getLogger(__name__)
    writer = SummaryWriter(LOG_DIR_TENSORBOARD)

    logger.info(f"--- LAUNCHING EXPERIMENT: {experiment_name} ---")

    all_configs = {k: v for k, v in globals().items() if k.isupper()}
    log_configurations(CURRENT_RUN_DIR, all_configs)
    log_environment(CURRENT_RUN_DIR)

    logger.info(f"--- Initializing FNetHybridTransformer ---")
    model = FNetHybridTransformer(
        num_encoder_layers=NUM_ENCODER_LAYERS,
        num_decoder_layers=NUM_DECODER_LAYERS,
        num_heads=NUM_HEADS,
        d_model=D_MODEL,
        dff=D_FF,
        vocab_size=VOCAB_SIZE,
        max_length=MAX_LENGTH,
        dropout=DROPOUT
    )

    model.apply(init_weights_fnet)
    nn.init.normal_(model.pos_embedding.weight, mean=0.0, std=0.02)
    model.final_linear.weight = model.embedding.weight

    model.to(device)
    count_parameters(model)

    # 4. SETUP OPTIMIZER
    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LEARNING_RATE, betas=(0.9, 0.98),
                                  eps=1e-9, weight_decay=WEIGHT_DECAY)

    # Scheduler
    scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=WARMUP_STEPS,
                                                num_training_steps=TARGET_TRAINING_STEPS)
    scaler = torch.cuda.amp.GradScaler()

# --- AUTO-RESUME LOGIC (SMARTER VERSION) ---
    global_step = 0
    best_bleu = 0.0
    LAST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "last.pt")
    BEST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "best.pt")

    # 1. Try to find the latest checkpoint (if it exists)
    if os.path.exists(LAST_CHECKPOINT_PATH):
        logger.info(f"🔄 Found checkpoint at {LAST_CHECKPOINT_PATH}. Resuming...")
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        global_step = checkpoint['global_step']
        best_bleu = checkpoint.get('best_bleu', 0.0)
        logger.info(f"   ✅ Resumed from Step {global_step} (LAST)")

    # 2. If no LAST, try to find the BEST checkpoint (Fall back to this!)
    elif os.path.exists(BEST_CHECKPOINT_PATH):
        logger.info(f"🔙 'last.pt' not found. Falling back to BEST checkpoint: {BEST_CHECKPOINT_PATH}")
        checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        global_step = checkpoint['global_step']
        best_bleu = checkpoint.get('best_bleu', 0.0)
        logger.info(f"   ✅ Resumed from Step {global_step} (BEST)")

    # 3. Start Fresh
    else:
        logger.info("🆕 No checkpoint found. Starting fresh training.")
    # 5. TRAINING LOOP
    model.train()

    # Resume progress bar from global_step
    progress_bar = tqdm(total=TARGET_TRAINING_STEPS, initial=global_step, desc="Training Steps")
    training_complete = False

    # Initialize gradients
    optimizer.zero_grad(set_to_none=True)

    # We iterate until global_step reaches the target
    epoch = 0
    while not training_complete:
        train_dataloader.generator.manual_seed(SEED + epoch)
        epoch += 1

        for batch_idx, batch in enumerate(train_dataloader):
            if global_step >= TARGET_TRAINING_STEPS:
                training_complete = True
                break

            input_ids = batch['input_ids'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            decoder_start_token = torch.full((labels.shape[0], 1), tokenizer.pad_token_id, dtype=torch.long, device=device)
            decoder_input_ids = torch.cat([decoder_start_token, labels[:, :-1]], dim=1)
            decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
            target_labels = labels

            src_padding_mask, tgt_padding_mask, mem_key_padding_mask, tgt_mask = model.create_masks(input_ids, decoder_input_ids)
            tgt_padding_mask[:, 0] = False

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                model_outputs = model(src=input_ids, tgt=decoder_input_ids, src_padding_mask=src_padding_mask,
                                      tgt_padding_mask=tgt_padding_mask, memory_key_padding_mask=mem_key_padding_mask,
                                      tgt_mask=tgt_mask)
                loss, loss_components = calculate_combined_loss(model_outputs, target_labels)

                # --- GRADIENT ACCUMULATION SCALING ---
                loss = loss / GRAD_ACCUMULATION_STEPS

            # Accumulate gradients (no optimizer step yet)
            scaler.scale(loss).backward()

            # --- OPTIMIZER STEP (Conditional) ---
            if (batch_idx + 1) % GRAD_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                # Reset gradients
                optimizer.zero_grad(set_to_none=True)

                global_step += 1
                progress_bar.update(1)
                lr = scheduler.get_last_lr()[0]

                if global_step % 20 == 0:
                    # Scale loss back up for logging purposes
                    logged_loss = loss.item() * GRAD_ACCUMULATION_STEPS
                    writer.add_scalar('train/loss', logged_loss, global_step)
                    writer.add_scalar('train/learning_rate', lr, global_step)
                    writer.add_scalar('train/gradient_norm', total_grad_norm.item(), global_step)
                    progress_bar.set_postfix(
                        loss=f"{logged_loss:.2f}",
                        lr=f"{lr:.2e}",
                        grad=f"{total_grad_norm.item():.2f}"  # Showing Gradients
                    )

                # --- PERIODIC SAVING (Every 500 Steps) ---
                # Saves you if Colab crashes mid-epoch
                if global_step % 500 == 0:
                     torch.save({
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'scaler_state_dict': scaler.state_dict(),
                        'best_bleu': best_bleu
                    }, LAST_CHECKPOINT_PATH)

                # --- VALIDATION CHECK ---
                if global_step in VALIDATION_SCHEDULE:
                    logger.info(f"\n--- Validation at Step {global_step} ---")
                    bleu_score = evaluate(model, val_dataloader, device)
                    writer.add_scalar('validation/bleu', bleu_score, global_step)
                    logger.info(f"Validation BLEU: {bleu_score:.4f} (Best: {best_bleu:.4f})")
                    #generate_sample_translations(model, device, sample_sentences_de_for_tracking)

                    if bleu_score > best_bleu:
                        best_bleu = bleu_score
                        logger.info(f"  New best BLEU! Saving best model...")
                        # Save EVERYTHING so you can resume even from best model
                        torch.save({
                            'global_step': global_step,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'scheduler_state_dict': scheduler.state_dict(),
                            'scaler_state_dict': scaler.state_dict(),
                            'best_bleu': best_bleu
                        }, BEST_CHECKPOINT_PATH)

                    model.train()

    progress_bar.close()
    writer.close()

    # Save Final (With States)
    torch.save({
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_bleu': best_bleu
    }, LAST_CHECKPOINT_PATH)

    print("\n" + "*"*80)
    print(" EXPERIMENT COMPLETE ")
    print("*"*80)

In [ ]:
import os
import sys
import torch
import transformers
import datasets
import torchmetrics
import numpy
import pkg_resources

def log_environment_separate(log_dir):
    # Define the separate file path
    meta_file = os.path.join(log_dir, "system_metadata.txt")

    with open(meta_file, "w") as f:
        # --- PART 1: SUMMARY ---
        f.write("="*40 + "\n")
        f.write("CORE ENVIRONMENT SUMMARY\n")
        f.write("="*40 + "\n")
        f.write(f"Python:       {sys.version.split()[0]}\n")
        f.write(f"PyTorch:      {torch.__version__}\n")
        f.write(f"Transformers: {transformers.__version__}\n")
        f.write(f"Datasets:     {datasets.__version__}\n")
        f.write(f"TorchMetrics: {torchmetrics.__version__}\n")
        f.write(f"NumPy:        {numpy.__version__}\n")

        try:
            import sacrebleu
            f.write(f"SacreBLEU:    {sacrebleu.__version__}\n")
        except ImportError:
            f.write("SacreBLEU:    Not Installed\n")

        if torch.cuda.is_available():
            f.write(f"GPU Name:     {torch.cuda.get_device_name(0)}\n")
            f.write(f"CUDA Ver:     {torch.version.cuda}\n")
            f.write(f"Capability:   {torch.cuda.get_device_capability(0)}\n")
        else:
            f.write("GPU:          None (CPU Only)\n")

        # --- PART 2: FULL FREEZE ---
        f.write("\n" + "="*40 + "\n")
        f.write("FULL LIBRARY DEPENDENCIES (PIP FREEZE)\n")
        f.write("="*40 + "\n")

        installed_packages = {d.project_name: d.version for d in pkg_resources.working_set}
        for package, version in sorted(installed_packages.items()):
            f.write(f"{package}=={version}\n")

    print(f"✅ Environment details saved SEPARATELY to: {meta_file}")

# Execute
# Assumes CURRENT_RUN_DIR is defined from your config
log_environment_separate(CURRENT_RUN_DIR)

In [ ]:
# TENSORBOARD VISUALIZATION

%load_ext tensorboard

TENSORBOARD_BASE_DIR = os.path.join(DRIVE_BASE_PATH)

%tensorboard --logdir "{TENSORBOARD_BASE_DIR}"

In [ ]:
from google.colab import runtime
runtime.unassign()

## End